# 02 · First simple model

Phase 1, Nov 6–7. Question: **is this gesture BFRB-like or ordinary?** Motion data only.

How it works:

1. For each reading, compute features that the MPU6050 can also produce (see `saha/features.py`): wrist tilt, acceleration without gravity, and rotation speed.
2. Summarise each recording's **gesture part** into one row: the mean, spread, minimum and maximum of each feature.
3. Train a gradient-boosted tree model and test it on **people it has never seen** (5 folds, split by `subject`).

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

from saha.features import ACC, ROT, add_channels, summarise

DATA = "../data/cmi-detect-behavior-with-sensor-data"
LABELS = ["sequence_id", "subject", "sequence_type", "orientation", "phase", "gesture"]
df = pd.read_csv(f"{DATA}/train.csv", usecols=LABELS + ACC + ROT)

## 1. One row of features per recording

In [ ]:
rows = []
for sid, seq in df.groupby("sequence_id", sort=False):
    g = add_channels(seq)
    g = g[g.phase == "Gesture"]
    if len(g) < 5:  # recordings with no gesture part
        continue
    feats = summarise(g).to_dict()
    feats["sequence_id"] = sid
    for c in ["subject", "sequence_type", "orientation", "gesture"]:
        feats[c] = seq[c].iloc[0]
    rows.append(feats)

data = pd.DataFrame(rows)
FEATURES = [c for c in data.columns if c not in ("sequence_id", "subject", "sequence_type", "orientation", "gesture")]
X = data[FEATURES]
y = (data.sequence_type == "Target").astype(int)  # 1 = BFRB-like
groups = data.subject
print(len(data), "recordings,", len(FEATURES), "features,", groups.nunique(), "people")

## 2. Test on unseen people

Each fold holds out about 16 people. The model never sees their data while training.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
prob = np.zeros(len(data))
for train, test in cv.split(X, y, groups):
    assert not set(groups.iloc[train]) & set(groups.iloc[test])
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)
    model.fit(X.iloc[train], y.iloc[train])
    prob[test] = model.predict_proba(X.iloc[test])[:, 1]
pred = (prob >= 0.5).astype(int)

print(f"balanced accuracy: {balanced_accuracy_score(y, pred):.3f}   (0.5 = guessing)")
print(f"F1 for BFRB-like:  {f1_score(y, pred):.3f}")
print(f"ROC AUC:           {roc_auc_score(y, prob):.3f}   (0.5 = guessing)")
print(f"always saying 'BFRB-like' would be right {y.mean():.1%} of the time")

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=["ordinary", "BFRB-like"]).plot(cmap="Blues", colorbar=False)
plt.title("Unseen people, all 5 folds");

## 3. Where does it go wrong?

Share of each gesture that the model calls BFRB-like. Ideally BFRB-like gestures are near 100% and ordinary ones near 0%.

In [ ]:
data["called_bfrb"] = pred
(data.groupby(["sequence_type", "gesture"]).called_bfrb.mean().mul(100).round(1)
     .sort_values(ascending=False).rename("% called BFRB-like").to_frame())

And by body position, using balanced accuracy:

In [ ]:
data.groupby("orientation").apply(
    lambda d: round(balanced_accuracy_score(d.sequence_type == "Target", d.called_bfrb), 3), include_groups=False
).rename("balanced accuracy").to_frame()

## 4. Score log

Write the score down here and in your notes each time the model changes.

| Date | Change | Balanced accuracy | ROC AUC |
|---|---|---|---|
| Nov 1 | First model: gesture part, 36 summary features, unseen-people test | 0.886 | 0.957 |

## Limits of this first model

- It looks only at the **gesture part**, which the labels mark for us. On the wearable nobody marks where a gesture starts, so the next version should slide a window along the whole recording.
- Left-handed people wore the device on their left wrist, which mirrors some axes. They are 10 of 81 people, so this is a small effect for now.